# argentina.personas — Pruebas interactivas

Recorrido paso a paso del módulo `argentina.personas`.

Funciones simples para limpiar/validar identificadores (DNI, CUIT/CUIL) y normalizar nombres. Solo stdlib — sin APIs externas, sin datasets internos, sin detección de sexo.

## 1. Setup e imports

In [1]:
import argentina as arg

print(f"argentina v{arg.__version__}")

argentina v0.0.23


## 2. DNI

`limpiar_dni` saca todo lo que no sea dígito. `validar_dni` chequea que queden 7 u 8 dígitos.

In [2]:
# Caso típico: DNI con puntos
arg.personas.limpiar_dni("12.345.678")

'12345678'

In [3]:
# Espacios y otros separadores también se sacan
arg.personas.limpiar_dni(" 12 345 678 ")

'12345678'

In [4]:
# Acepta int directo
arg.personas.limpiar_dni(12345678)

'12345678'

In [5]:
# Casos sin dígitos → None
print(arg.personas.limpiar_dni(None))
print(arg.personas.limpiar_dni(""))
print(arg.personas.limpiar_dni("abc"))

None
None
None


In [6]:
# Validación: 7 u 8 dígitos
print(arg.personas.validar_dni("12.345.678"))   # 8 dígitos
print(arg.personas.validar_dni("1.234.567"))    # 7 dígitos (DNIs viejos)
print(arg.personas.validar_dni("123"))          # corto
print(arg.personas.validar_dni("123456789"))    # largo
print(arg.personas.validar_dni(None))

True
True
False
False
False


## 3. CUIT / CUIL

`limpiar_cuit` deja solo dígitos. `validar_cuit` chequea **longitud (11) + dígito verificador** (algoritmo oficial). Para validar solo formato sin chequear dígito, pasá `digito=False`.

In [7]:
arg.personas.limpiar_cuit("20-12345678-3")

'20123456783'

In [8]:
# Otras formas de escribir el mismo CUIT
print(arg.personas.limpiar_cuit("20.12345678.3"))
print(arg.personas.limpiar_cuit("20 12345678 3"))
print(arg.personas.limpiar_cuit("20123456783"))

20123456783
20123456783
20123456783


In [9]:
# Validación completa (dígito verificador correcto)
print(arg.personas.validar_cuit("20-12345678-6"))   # dígito correcto → True
print(arg.personas.validar_cuit("20-12345678-3"))   # dígito incorrecto → False
print(arg.personas.validar_cuit("20-123"))           # incompleto → False
print(arg.personas.validar_cuit(None))

# Si solo te interesa el largo (back-compat)
print(arg.personas.validar_cuit("20-12345678-3", digito=False))   # solo largo → True

True
False
False
False
True


## 4. Extraer DNI desde CUIT

Estructura del CUIT/CUIL: `XX-DDDDDDDD-V` (2 prefijo + 8 dígitos del DNI + 1 verificador).

In [10]:
arg.personas.extraer_dni_de_cuit("20-12345678-3")

'12345678'

In [11]:
# Distintos prefijos (20/23/24/27 personas físicas, 30/33 personas jurídicas)
print(arg.personas.extraer_dni_de_cuit("27-31234567-4"))
print(arg.personas.extraer_dni_de_cuit("23-40123456-9"))
print(arg.personas.extraer_dni_de_cuit("30-71234567-1"))   # también extrae el 'DNI' aunque sea jurídica

31234567
40123456
71234567


In [12]:
# Si no son 11 dígitos → None
print(arg.personas.extraer_dni_de_cuit("20-123"))
print(arg.personas.extraer_dni_de_cuit(None))
print(arg.personas.extraer_dni_de_cuit(""))

None
None
None


## 5. calcular_digito_cuit

Implementa el algoritmo oficial: multiplicadores `5 4 3 2 7 6 5 4 3 2` sobre los primeros 10 dígitos, módulo 11, con wrap-around `10→9` y `11→0`.

In [13]:
# Caso típico
arg.personas.calcular_digito_cuit("2012345678")

'6'

In [14]:
# Roundtrip: si calculás el dígito de los primeros 10 de un CUIT válido, te da el 11° dígito
cuit = "20123456786"
primeros_10 = cuit[:10]
ultimo = cuit[-1]
esperado = arg.personas.calcular_digito_cuit(primeros_10)
print(f"CUIT:       {cuit}")
print(f"esperado:   {esperado}")
print(f"ultimo:     {ultimo}")
print(f"matchea:    {esperado == ultimo}")

CUIT:       20123456786
esperado:   6
ultimo:     6
matchea:    True


In [15]:
# Si no le pasás exactamente 10 dígitos → None
print(arg.personas.calcular_digito_cuit("201234567"))     # 9
print(arg.personas.calcular_digito_cuit("20123456789"))   # 11
print(arg.personas.calcular_digito_cuit(None))
print(arg.personas.calcular_digito_cuit("abc"))

None
None
None
None


## 6. tipo_cuit

Clasifica según prefijo:
- `20` / `23` / `24` / `27` → persona física
- `30` / `33` / `34` → persona jurídica
- otros → `None`

In [16]:
for cuit in [
    "20-12345678-6",   # 20 → física (M)
    "27-31234567-4",   # 27 → física (F)
    "23-40123456-9",   # 23 → física (ambiguo)
    "30-71234567-1",   # 30 → jurídica
    "33-71234567-9",   # 33 → jurídica (IVA)
    "50-12345678-1",   # prefijo no listado → None
    "20-123",            # incompleto → None
    None,
]:
    print(f"{cuit!s:18} → {arg.personas.tipo_cuit(cuit)!r}")

20-12345678-6      → 'persona_fisica'
27-31234567-4      → 'persona_fisica'
23-40123456-9      → 'persona_fisica'
30-71234567-1      → 'persona_juridica'
33-71234567-9      → 'persona_juridica'
50-12345678-1      → None
20-123             → None
None               → None


## 7. Formato canónico: formatear_dni / formatear_cuit

`formatear_dni` agrega puntos de miles. `formatear_cuit` produce `XX-XXXXXXXX-X`.

In [17]:
for dni in ["12345678", "1234567", "123", None, ""]:
    print(f"{dni!s:12} → {arg.personas.formatear_dni(dni)!r}")

12345678     → '12.345.678'
1234567      → '1.234.567'
123          → None
None         → None
             → None


In [18]:
for cuit in ["20123456786", "20-12345678-6", "27.31234567.4", "20-123", None]:
    print(f"{cuit!s:18} → {arg.personas.formatear_cuit(cuit)!r}")

20123456786        → '20-12345678-6'
20-12345678-6      → '20-12345678-6'
27.31234567.4      → '27-31234567-4'
20-123             → None
None               → None


## 5. normalizar_nombre

Lowercase, sin tildes, espacios colapsados, sólo letras (preserva `ñ`).

In [19]:
arg.personas.normalizar_nombre(" María   Laura ")

'maria laura'

In [20]:
# Conserva la ñ
arg.personas.normalizar_nombre("Iñaki Núñez")

'inaki nunez'

In [21]:
# Saca dígitos, signos y comas
arg.personas.normalizar_nombre("Pérez, Juan Carlos (h)")

'perez juan carlos h'

In [22]:
# Casos vacíos / sin letras → None
print(arg.personas.normalizar_nombre(None))
print(arg.personas.normalizar_nombre(""))
print(arg.personas.normalizar_nombre("   "))
print(arg.personas.normalizar_nombre("123 ---"))

None
None
None
None


## 6. primer_nombre / apellido_principal

Atajos sobre `normalizar_nombre` para quedarse con el primer token.

In [23]:
print(arg.personas.primer_nombre("María Laura"))
print(arg.personas.primer_nombre("Juan Carlos Pérez"))
print(arg.personas.primer_nombre("  Ana  "))

maria
juan
ana


In [24]:
print(arg.personas.apellido_principal("Pérez Gómez"))
print(arg.personas.apellido_principal("García"))
print(arg.personas.apellido_principal("D'Onofrio Martínez"))   # la comilla se trata como separador

perez
garcia
d


In [25]:
# Igual que normalizar_nombre, devuelven None cuando no hay letras
print(arg.personas.primer_nombre(None))
print(arg.personas.apellido_principal(""))

None
None


## 7. Combinando todo

Pipeline típico: dado un CUIT y un nombre completo, obtener el DNI y los componentes normalizados.

In [26]:
registros = [
    ("20-12345678-6", "María Laura Pérez Gómez"),     # CUIT válido
    ("27-31234567-3", " Juan Carlos D'Onofrio "),       # dígito incorrecto
    ("30-71234567-1", "Empresa S.A."),                  # persona jurídica
    ("20-123",         "José Luis"),                     # CUIT incompleto
]

for cuit, nombre in registros:
    print({
        "cuit_valido":    arg.personas.validar_cuit(cuit),
        "tipo":           arg.personas.tipo_cuit(cuit),
        "cuit_formato":   arg.personas.formatear_cuit(cuit),
        "dni":            arg.personas.extraer_dni_de_cuit(cuit),
        "dni_formato":    arg.personas.formatear_dni(
            arg.personas.extraer_dni_de_cuit(cuit)
        ),
        "primer_nombre":  arg.personas.primer_nombre(nombre),
        "apellido":       arg.personas.apellido_principal(nombre),
    })

{'cuit_valido': True, 'tipo': 'persona_fisica', 'cuit_formato': '20-12345678-6', 'dni': '12345678', 'dni_formato': '12.345.678', 'primer_nombre': 'maria', 'apellido': 'maria'}
{'cuit_valido': False, 'tipo': 'persona_fisica', 'cuit_formato': '27-31234567-3', 'dni': '31234567', 'dni_formato': '31.234.567', 'primer_nombre': 'juan', 'apellido': 'juan'}
{'cuit_valido': True, 'tipo': 'persona_juridica', 'cuit_formato': '30-71234567-1', 'dni': '71234567', 'dni_formato': '71.234.567', 'primer_nombre': 'empresa', 'apellido': 'empresa'}
{'cuit_valido': False, 'tipo': None, 'cuit_formato': None, 'dni': None, 'dni_formato': None, 'primer_nombre': 'jose', 'apellido': 'jose'}


## 8. Estimar año de nacimiento desde el DNI

**Modelo**: función lineal calibrada contra dos hitos públicos del RENAPER.

- DNI 59.999.999 = agosto 2023 (último pre-salto)
- Pendiente ~736.470 DNI/año (RENAPER emite ~750k DNI/año: nacimientos + naturalizaciones + extranjeros + reasignaciones)

Resultado: **`año = 1942.20 + DNI / 736470`** (pre-salto), continuada después del salto a 70.000.000 con la misma pendiente.

Es consistente con la [fórmula viral de Reddit/X](https://x.com/HernaniiBA/status/1909951632592216551) (`1942.5 + DNI/736470`) y con las tablas que circulan en blogs de calculadoras de edad.

**Atajo cronológico clave**: la franja 60.000.000–69.999.999 NO se asigna como DNI personal — se reservó en dic-2019 (Disposición Renaper 4678/2019) para CUIT/CUIL provisorios de extranjeros.

### 8.1. estimar_año_nacimiento

In [27]:
# Recorrido completo del rango
for dni in ["5.000.000", "15.000.000", "25.000.000", "35.000.000",
            "45.000.000", "55.000.000", "70.500.000", "72.000.000"]:
    print(f"DNI {dni}  →  {arg.personas.estimar_año_nacimiento(dni)}")

DNI 5.000.000  →  1948
DNI 15.000.000  →  1962
DNI 25.000.000  →  1976
DNI 35.000.000  →  1989
DNI 45.000.000  →  2003
DNI 55.000.000  →  2016
DNI 70.500.000  →  2024
DNI 72.000.000  →  2026


In [28]:
# Franja CUIT extranjero → None
for dni in [60_000_000, 65_000_000, 69_999_999]:
    print(f"DNI {dni:,}: {arg.personas.estimar_año_nacimiento(dni)}".replace(",", "."))

DNI 60.000.000: None
DNI 65.000.000: None
DNI 69.999.999: None


In [29]:
# Continuidad cronológica al salto
print("DNI 59.999.999 (último pre-salto, agosto 2023):",
      arg.personas.estimar_año_nacimiento(59_999_999))
print("DNI 70.000.001 (primer post-salto, sept 2023):",
      arg.personas.estimar_año_nacimiento(70_000_001))

DNI 59.999.999 (último pre-salto, agosto 2023): 2023
DNI 70.000.001 (primer post-salto, sept 2023): 2023


In [30]:
# Inválidos / fuera de rango razonable
print(arg.personas.estimar_año_nacimiento("abc"))
print(arg.personas.estimar_año_nacimiento("123"))
print(arg.personas.estimar_año_nacimiento(99_999_999))   # fuera, devuelve None
print(arg.personas.estimar_año_nacimiento(None))

None
None
None
None


### 8.2. estimar_dni (inverso)

In [31]:
for año in [1950, 1970, 1985, 1995, 2005, 2020, 2024]:
    dni = arg.personas.estimar_dni(año)
    print(f"{año}  →  DNI ~{dni:,}".replace(",", "."))

1950  →  DNI ~6.114.700
1970  →  DNI ~20.844.100
1985  →  DNI ~31.891.150
1995  →  DNI ~39.255.850
2005  →  DNI ~46.620.550
2020  →  DNI ~57.667.600
2024  →  DNI ~70.613.480


### 8.3. Rango de DNIs por año

In [32]:
for año in [1970, 1990, 2000, 2015, 2024]:
    rango = arg.personas.rango_dni_de_año(año)
    inicio, fin = rango
    ancho = fin - inicio + 1
    print(f"{año}: {inicio:>11,} .. {fin:>11,}  ({ancho:>9,} DNIs)".replace(",", "."))

1970:  20.475.865 ..  21.212.334  (  736.470 DNIs)
1990:  35.205.265 ..  35.941.734  (  736.470 DNIs)
2000:  42.569.965 ..  43.306.434  (  736.470 DNIs)
2015:  53.617.015 ..  54.353.484  (  736.470 DNIs)
2024:  70.245.245 ..  70.981.714  (  736.470 DNIs)


### 8.4. Pipeline: DNI → edad estimada

In [33]:
from datetime import date
hoy = date.today()

for dni in ["18.500.000", "27.000.000", "35.250.000", "48.000.000", "55.500.000"]:
    año = arg.personas.estimar_año_nacimiento(dni)
    edad = hoy.year - año if año else None
    print(f"DNI {dni}  →  nacido ~{año}  →  edad ~{edad}")

DNI 18.500.000  →  nacido ~1967  →  edad ~59
DNI 27.000.000  →  nacido ~1978  →  edad ~48
DNI 35.250.000  →  nacido ~1990  →  edad ~36
DNI 48.000.000  →  nacido ~2007  →  edad ~19
DNI 55.500.000  →  nacido ~2017  →  edad ~9


### 8.5. Datos auxiliares: serie histórica oficial de nacimientos

El paquete embebe la serie del [DEIS / Ministerio de Salud](https://datos.salud.gob.ar/dataset/serie-historica-de-nacimientos-ocurridos-en-argentina-por-jurisdiccion) (1914–2024) como referencia. **No se usa en el estimador** — el modelo lineal calibrado contra hitos del RENAPER da mejores resultados que un modelo basado solo en nacimientos (porque el RENAPER emite ~750k DNI/año mientras nacen ~500-700k chicos). Pero la serie está disponible si te sirve por otro motivo.

In [34]:
serie = arg.personas.serie_nacimientos()
print(f"{len(serie)} años de datos, {serie[0][0]}-{serie[-1][0]}")
print(f"Pico:   {max(serie, key=lambda x: x[1])}")
print(f"Último: {serie[-1]}")

111 años de datos, 1914-2024
Pico:   (2014, 777012)
Último: (2024, 413135)


### 8.6. Límites de la estimación

**Margen típico ±2-3 años**. Casos donde puede fallar mucho más:

- **Inscripciones tardías**: alguien que tramitó su primer DNI a los 20 años recibe el número de la cohorte que se inscribe ese año, no el de su nacimiento. (Por eso aparecen DNIs altos en personas mayores.)
- **Naturalizaciones**: los extranjeros que se hacen argentinos reciben DNI en orden de naturalización.
- **Pre-1968 (DNI < ~14M)**: el sistema de Libreta Cívica/Enrolamiento asignaba el número al inscribirse (~16-18 años o servicio militar), no al nacer. Para DNI bajos el desfase con el año real de nacimiento puede ser de varios años.
- **Para identificar a una persona específica esto no sirve.** Sirve para imputar año aproximado en datasets de DNIs anónimos o sintéticos.

## 9. Tests automáticos

Los tests viven en `tests/test_personas.py`. Para correrlos:

```bash
cd /Users/tobiasyatche/argentina
pytest tests/test_personas.py -v
```

## Notas sueltas / TODOs

- Solo stdlib (`re`, `unicodedata`, `csv`, `bisect`). Sin pandas, sin APIs externas.
- La validación de CUIT chequea formato + dígito verificador (algoritmo oficial). Para sólo formato pasá `digito=False`.
- `extraer_dni_de_cuit` no distingue entre persona física y jurídica: toma siempre los 8 dígitos centrales.
- Detección de sexo por nombre queda fuera del scope.
- `estimar_año_nacimiento`: tabla `data/dni_anclajes.csv` editable. Si encontrás puntos más confiables, agregalos al CSV y la interpolación los toma.